# Session 3 — Clinical Note Triage · Data Analysis & Problem Understanding

*Personal EDA companion to `clinical_note_triage.ipynb`. **It does not submit** — it
downloads the notes and inspects them so you understand the text-classification problem
before training.*

**The task.** Read a free-text **medical transcription** and predict which of **12 clinical
specialties** (`spec_01` … `spec_12`) it belongs to — automatic triage. Scored on
**F1-macro**, so rare specialties weigh as much as common ones.

What this notebook answers:
1. How many notes, how long are they, how balanced are the 12 specialties?
2. What's the F1-macro floor (majority guess)?
3. Which **words** characterise each specialty (the signal TF-IDF will use)?
4. Are the specialties separable in TF-IDF space — and which ones get confused?
5. Any data-quality issues (duplicates, train/test overlap, empty notes)?

## 0. Setup

In [ ]:
!pip -q install "mlarena-sdk==0.3.0" seaborn   # installs the `mlarena` package + plotting

import os, re, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid", context="notebook")

import mlarena
API_TOKEN = "mlk_user_REPLACE_ME"   # <-- paste your token (Profile page on ml-arena.com)
COMPETITION_ID = 174
client = mlarena.connect(api_key=API_TOKEN, base_url="https://ml-arena.com")

## 1. Download & load the data

`download_dataset` pulls `train.csv` (`id, transcription, label`) and `test.csv`
(`id, transcription`) into `data/`.

In [ ]:
if not os.path.exists("data/train.csv"):
    client.download_dataset(COMPETITION_ID, "data/")

train = pd.read_csv("data/train.csv")
test  = pd.read_csv("data/test.csv")
classes = sorted(train["label"].unique())

print("train:", train.shape, "  test:", test.shape)
print(f"{len(classes)} specialties: {classes}")
train.head(2)

## 2. Class balance & the F1-macro floor

The majority-guess baseline scores near-zero F1-macro because it abandons 11 of 12 classes.

In [ ]:
vc = train["label"].value_counts().sort_index()
print(vc.to_string())
print(f"\nimbalance ratio (max/min) = {vc.max() / vc.min():.1f}")

from sklearn.metrics import f1_score
maj = vc.idxmax()
print(f"F1-macro if you always predict '{maj}' = {f1_score(train['label'], [maj]*len(train), average='macro'):.4f}")

vc.plot.bar(rot=45, color="#4c72b0"); plt.title("Specialty distribution"); plt.ylabel("notes"); plt.show()

## 3. Note length

How long are the transcriptions, and does length vary by specialty? This drives the
truncation / chunking decision for any transformer (which has a token limit).

In [ ]:
train["n_words"] = train["transcription"].fillna("").str.split().str.len()
train["n_chars"] = train["transcription"].fillna("").str.len()
print(train[["n_words", "n_chars"]].describe().round(1).to_string())

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
train["n_words"].plot.hist(bins=50, ax=ax[0], color="#4c72b0"); ax[0].set_title("Words per note"); ax[0].set_xlabel("words")
sns.boxplot(data=train, x="label", y="n_words", ax=ax[1]); ax[1].set_title("Note length by specialty")
ax[1].tick_params(axis="x", rotation=45); plt.tight_layout(); plt.show()

## 4. What words characterise each specialty?

The signal a TF-IDF baseline exploits. For each specialty we rank terms by mean TF-IDF
weight — the vocabulary that makes that specialty distinctive.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vec = TfidfVectorizer(stop_words="english", min_df=5, ngram_range=(1, 2), max_features=40000)
Xtf = vec.fit_transform(train["transcription"].fillna(""))
terms = np.array(vec.get_feature_names_out())

print("Top terms per specialty (by mean TF-IDF):\n")
for cl in classes:
    mean = np.asarray(Xtf[(train["label"] == cl).values].mean(0)).ravel()
    top = terms[mean.argsort()[::-1][:12]]
    print(f"{cl:8s}: {', '.join(top)}")

In [ ]:
# Most common raw words overall (incl. stopwords) — a feel for shared boilerplate.
from collections import Counter
words = Counter()
for t in train["transcription"].fillna("").head(3000):
    words.update(re.findall(r"[a-z]{3,}", t.lower()))
pd.Series(dict(words.most_common(25))).iloc[::-1].plot.barh(figsize=(7, 6), color="#dd8452")
plt.title("Most common words (raw)"); plt.xlabel("count in first 3000 notes"); plt.show()

## 5. Are the specialties separable?

Project the TF-IDF vectors to 2-D with TruncatedSVD (LSA). Clusters that pull apart are
easy; specialties that overlap are the ones your model will confuse.

In [ ]:
from sklearn.decomposition import TruncatedSVD
sv = TruncatedSVD(2, random_state=0).fit_transform(Xtf)
sns.scatterplot(x=sv[:, 0], y=sv[:, 1], hue=train["label"], s=12, palette="tab20")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.title("TF-IDF → SVD(2)  (LSA projection)"); plt.tight_layout(); plt.show()

## 6. Data-quality checks

Duplicate notes, train/test overlap (leakage), and empty/stub transcriptions.

In [ ]:
print("duplicate transcriptions in train:", int(train["transcription"].duplicated().sum()))
print("train/test identical-note overlap :", len(set(train["transcription"]) & set(test["transcription"])))
print("very short notes (<5 words)       :", int((train["n_words"] < 5).sum()))
print("missing transcriptions            :", int(train["transcription"].isna().sum()))

## 7. The TF-IDF baseline — and which specialties confuse it

Reproduce the starter (TF-IDF + logistic regression) with cross-validated predictions, then
read the per-class report and confusion matrix. The off-diagonal shows which specialties
share vocabulary and need a smarter model to tell apart.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

model = Pipeline([
    ("tfidf", TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2),
                              min_df=3, max_features=40000, stop_words="english")),
    ("clf", LogisticRegression(max_iter=2000, C=5.0, class_weight="balanced")),
])
pred = cross_val_predict(model, train["transcription"].fillna(""), train["label"], cv=3)
print(f"CV F1-macro = {f1_score(train['label'], pred, average='macro'):.4f}\n")
print(classification_report(train["label"], pred))

In [ ]:
ConfusionMatrixDisplay(confusion_matrix(train["label"], pred, labels=classes), display_labels=classes)\
    .plot(xticks_rotation=45, cmap="Blues", colorbar=False)
plt.title("Confusion matrix — TF-IDF + LogReg (CV)"); plt.tight_layout(); plt.show()

## 8. Takeaways → modelling roadmap

- **The floor is ~0 F1-macro** (majority guess); TF-IDF + LogReg is already a strong baseline.
  Beating it means handling the overlapping specialties from Sections 5 & 7.
- **Rare specialties drive F1-macro** — watch the lowest per-class F1; `class_weight="balanced"`
  and stratified CV help, but the confused pairs need real semantics.
- **Vocabulary is the signal** (Section 4). Tune the TF-IDF (`ngram_range`, `min_df`,
  `max_features`) and `C` for cheap gains before reaching for transformers.
- **Mind note length** (Section 3) — long notes exceed transformer token limits; plan
  truncation or chunking.
- **Domain models win:** a clinical/biomedical transformer (BioBERT / ClinicalBERT) understands
  medical vocabulary a general model misses. Embeddings + a light classifier capture most of the
  gain cheaply; LoRA/PEFT fine-tuning goes further on a single GPU.
- **Trust CV** — submit only when the cross-validated F1-macro improves.

Back to the competition starter: `clinical_note_triage.ipynb`.